In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

In [2]:
from data import load_documents, create_chunks
from embedder import Embedder
from search import build_text_index, build_vector_index

documents = load_documents()
chunks = create_chunks(documents)

embedder = Embedder(
    "models/Xenova/all-MiniLM-L6-v2"
)

text_index = build_text_index(chunks)

vector_index = build_vector_index(
    chunks,
    embedder
)

print("Documents:", len(documents))
print("Chunks:", len(chunks))

2026-08-09 21:57:14.419414777 [W:onnxruntime:Default, device_discovery.cc:134 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Documents: 72
Chunks: 295


In [3]:
from search import hybrid_search

query = "How do I give the model access to tools?"

results = hybrid_search(
    text_index,
    vector_index,
    embedder,
    query
)

for result in results:
    print(
        result["filename"],
        "- start:",
        result["start"]
    )

01-agentic-rag/lessons/13-function-calling.md - start: 4000
01-agentic-rag/lessons/13-function-calling.md - start: 2000
01-agentic-rag/lessons/01-intro.md - start: 2000
01-agentic-rag/lessons/14-agentic-loop.md - start: 0
04-evaluation/lessons/02-ground-truth.md - start: 1000


In [4]:
from rag import rag

question = "How does function calling work?"

rag_result = rag(
    openai_client,
    text_index,
    vector_index,
    embedder,
    question
)

print(rag_result["answer"])
print("Input tokens:", rag_result["input_tokens"])

Function calling works like this:

1. You define a tool for the model using a JSON schema, not Python code.
   - The schema includes a `name`, `description`, and `parameters`.
   - The `description` is especially important because the model uses it to decide when to call the function.

2. You send the user question to the model along with the tool definition.

3. The model may return a `function_call` instead of a normal answer.
   - It also provides JSON arguments for the call.
   - It can rewrite the user’s question into a better search query.

4. Your code executes the function.
   - You parse the arguments.
   - Run the actual Python function.
   - Send the result back in a `function_call_output` message.

5. The model uses that result to produce the final answer.
   - If needed, it can call tools again.
   - In an agent loop, you keep repeating this until the model returns a message with no more function calls.

So in short: the model decides when to call a tool and what arguments

In [5]:
from agent import agent_loop

question = "How does the agentic loop work, and how is it different from plain RAG?"

agent_result = agent_loop(
    openai_client,
    text_index,
    vector_index,
    embedder,
    question
)

print(agent_result["answer"])
print("Search calls:", agent_result["search_calls"])

Iteration #1
Function call: search {"query":"agentic loop plain RAG lesson agentic loop how does it work difference"}
Function call: search {"query":"LLM Zoomcamp agentic loop vs RAG retrieval generation iteration"}
Function call: search {"query":"course lesson agentic RAG agentic loop tools planning memory"}
Function call: search {"query":"plain RAG limitations agentic loop course"}
Iteration #2
The agentic loop is basically a `while` loop around the LLM:

1. Send the user question plus the current message history to the model.
2. If the model returns a tool/function call, execute it.
3. Append the tool result back into the history.
4. Call the model again.
5. Stop only when the model returns a final answer with no more tool calls.

In the course, this is described as the model being “in the driver’s seat”: the model decides whether it needs to search again, and your code just keeps the loop going until it’s done. The loop also keeps memory, meaning all prompts, model outputs, and too